# Checking support before reweighting

## Target-Positive Source Support

Let $n_{ag}^S$ be the source count receiving treatment level $a$ in stratum $g$. The released calculation requires

$$
\pi_g^T>0\quad\Longrightarrow\quad n_{1g}^S>0\ \text{ and }\ n_{0g}^S>0.
$$

The target probability $\pi_g^T$ determines whether the stratum contributes. The counts $n_{1g}^S$ and $n_{0g}^S$ determine whether both source conditional means can be supported. This numerical rule prevents extrapolation into a target-relevant stratum that lacks one source treatment group.

### Comparing coarse partitions with fine partitions

The public panel has support under the approved three-stratum rule. Finer rules can create target-positive strata with a missing source treatment group. The support audit is therefore performed before any target effect is calculated.

In [1]:
from pathlib import Path
import hashlib
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)


BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset
FINANCIAL_ROOT = BOOK_ROOT

import RiskLabAI
from RiskLabAI.causal_factor_analysis import transported_ate

assert RiskLabAI.__version__ == "3.2.0"

FIGURES = BOOK_ROOT / "outputs" / "figures" / "06_transportability"
TABLES = BOOK_ROOT / "outputs" / "tables" / "06_transportability"
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "06_transportability"
DATA_FILE = ensure_dataset(BOOK_ROOT, "mpdta")
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "violet": "#A03A78",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 11.5,
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["axes"],
    "axes.edgecolor": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "text.color": PALETTE["text"],
    "grid.color": PALETTE["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": PALETTE["axes"],
    "legend.edgecolor": PALETTE["grid"],
    "savefig.facecolor": PALETTE["background"],
    "savefig.transparent": False,
    "savefig.dpi": 300,
})


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (bool, np.bool_)):
        return "Pass" if bool(value) else "Fail"
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        value = float(value)
        if value == 0:
            return "0"
        if value.is_integer():
            return f"{int(value):,}"
        if abs(value) < 0.0001 or abs(value) >= 1000000:
            return f"{value:.6g}"
        return f"{value:.6f}".rstrip("0").rstrip(".")
    return str(value)


def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    latex = latex.replace("\\begin{table}[htbp]", "\\begin{table}[htbp]\n\\centering", 1)
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def finish_axis(axis, grid=True):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)


def save_figure(figure, stem):
    figure.savefig(FIGURES / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    figure.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    display(figure)
    plt.close(figure)


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def county_frame():
    raw = pd.read_csv(DATA_FILE)
    employment = raw.pivot(index="countyreal", columns="year", values="lemp")
    population = raw.pivot(index="countyreal", columns="year", values="lpop")
    first_treatment = raw.groupby("countyreal")["first_treat"].first()
    frame = pd.DataFrame({
        "County identifier": employment.index.to_numpy(),
        "Treatment group": (first_treatment.reindex(employment.index).to_numpy() > 0).astype(int),
        "Log employment in 2003": employment[2003].to_numpy(),
        "Log employment in 2007": employment[2007].to_numpy(),
        "Log population in 2003": population[2003].to_numpy(),
    })
    frame["Employment change, 2003 to 2007"] = frame["Log employment in 2007"] - frame["Log employment in 2003"]
    population_median = float(frame["Log population in 2003"].median())
    frame["Domain"] = np.where(frame["Log population in 2003"] <= population_median, "Source", "Target")
    frame["Stratum"] = pd.qcut(frame["Log employment in 2003"], 3, labels=False, duplicates="drop").astype(int)
    frame["Stratum name"] = frame["Stratum"].map({0: "Low baseline employment", 1: "Middle baseline employment", 2: "High baseline employment"})
    return raw, frame, population_median


def transport_summary(frame, stratum_column):
    strata = sorted(frame[stratum_column].dropna().unique())
    source = frame[frame["Domain"] == "Source"]
    target = frame[frame["Domain"] == "Target"]
    source_counts = pd.crosstab(source[stratum_column], source["Treatment group"]).reindex(index=strata, columns=[0, 1], fill_value=0)
    source_means = source.groupby([stratum_column, "Treatment group"])["Employment change, 2003 to 2007"].mean().unstack().reindex(index=strata, columns=[0, 1])
    source_sizes = source[stratum_column].value_counts().reindex(strata, fill_value=0)
    target_sizes = target[stratum_column].value_counts().reindex(strata, fill_value=0)
    result = pd.DataFrame({
        "Stratum": strata,
        "Source control count": source_counts[0].to_numpy(),
        "Source treated count": source_counts[1].to_numpy(),
        "Source control mean": source_means[0].to_numpy(),
        "Source treated mean": source_means[1].to_numpy(),
        "Source contrast": (source_means[1] - source_means[0]).to_numpy(),
        "Source probability": (source_sizes / source_sizes.sum()).to_numpy(),
        "Target count": target_sizes.to_numpy(),
        "Target probability": (target_sizes / target_sizes.sum()).to_numpy(),
    })
    return result


def capture_failure(name, calculation):
    try:
        calculation()
        return {"Check": name, "Rejected": False, "Message": "No error"}
    except ValueError as error:
        return {"Check": name, "Rejected": True, "Message": str(error)}

raw, counties, population_median = county_frame()
partition_rows = []
partition_details = {}
for requested_strata in range(2, 11):
    partitioned = counties.copy()
    partitioned["Partition stratum"] = pd.qcut(partitioned["Log employment in 2003"], requested_strata, labels=False, duplicates="drop").astype(int)
    strata = sorted(partitioned["Partition stratum"].unique())
    source = partitioned[partitioned["Domain"] == "Source"]
    target = partitioned[partitioned["Domain"] == "Target"]
    source_counts = pd.crosstab(source["Partition stratum"], source["Treatment group"]).reindex(index=strata, columns=[0, 1], fill_value=0)
    target_counts = target["Partition stratum"].value_counts().reindex(strata, fill_value=0)
    target_positive = target_counts > 0
    unsupported = target_positive & ((source_counts[0] == 0) | (source_counts[1] == 0))
    detail = pd.DataFrame({
        "Stratum": strata,
        "Target count": target_counts.to_numpy(),
        "Source control count": source_counts[0].to_numpy(),
        "Source treated count": source_counts[1].to_numpy(),
        "Target positive": target_positive.to_numpy(),
        "Supported": (~unsupported).to_numpy(),
    })
    partition_details[requested_strata] = detail
    positive_detail = detail[detail["Target positive"]]
    partition_rows.append({
        "Requested strata": requested_strata,
        "Realized strata": len(strata),
        "Target-positive strata": int(target_positive.sum()),
        "Supported target strata": int((target_positive & ~unsupported).sum()),
        "Unsupported target strata": int(unsupported.sum()),
        "Smallest source control count": int(positive_detail["Source control count"].min()),
        "Smallest source treated count": int(positive_detail["Source treated count"].min()),
        "Calculation status": "Pass" if not unsupported.any() else "Stop",
    })

partition_summary = pd.DataFrame(partition_rows)
save_table(partition_summary, "support_partition_summary", "Support audit as the public baseline partition becomes finer.", "tab:cim6-support-partition-summary", fit=True)
partition_book = partition_summary[["Requested strata", "Supported target strata", "Unsupported target strata", "Smallest source treated count", "Calculation status"]].copy()
save_table(partition_book, "support_partition_summary_book", "Readable support audit as the public partition becomes finer.", "tab:cim6-support-partition-summary-book", fit=True)

primary_support = partition_details[3].copy()
primary_support["Stratum name"] = primary_support["Stratum"].map({0: "Low", 1: "Middle", 2: "High"})
primary_support = primary_support[["Stratum", "Stratum name", "Target count", "Source control count", "Source treated count", "Target positive", "Supported"]]
save_table(primary_support, "support_primary_terciles", "Support audit for the approved three-stratum public calculation.", "tab:cim6-support-primary-terciles", fit=True)

quartile_support = partition_details[4].copy()
save_table(quartile_support, "support_quartile_audit", "Four-stratum audit showing the first public support failure.", "tab:cim6-support-quartile-audit", fit=True)

partitioned = counties.copy()
partitioned["Partition stratum"] = pd.qcut(partitioned["Log employment in 2003"], 4, labels=False, duplicates="drop").astype(int)
quartile_summary = transport_summary(partitioned, "Partition stratum")
quartile_failure = capture_failure(
    "Four-stratum target standardization",
    lambda: transported_ate(
        quartile_summary["Source treated mean"].fillna(0),
        quartile_summary["Source control mean"].fillna(0),
        quartile_summary["Target probability"],
        quartile_summary["Source treated count"],
        quartile_summary["Source control count"],
    ),
)
failure_record = pd.DataFrame([quartile_failure])
save_table(failure_record, "support_public_failure_record", "Recorded failure for the unsupported four-stratum public calculation.", "tab:cim6-support-public-failure", fit=True)
assert quartile_failure["Rejected"]

partition_summary.to_csv(FIGURE_DATA / "support_partition_counts.csv", index=False)
figure, axis = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
axis.plot(partition_summary["Requested strata"], partition_summary["Supported target strata"], marker="o", color=PALETTE["green"], linewidth=2.2, label="Supported target strata")
axis.plot(partition_summary["Requested strata"], partition_summary["Unsupported target strata"], marker="o", color=PALETTE["rose"], linewidth=2.2, label="Unsupported target strata")
axis.set_xlabel("Requested number of strata")
axis.set_ylabel("Number of target-positive strata")
axis.set_title("Finer Partitions Expose Public Support Gaps")
axis.legend()
finish_axis(axis)
save_figure(figure, "support_partition_counts")

primary_support.to_csv(FIGURE_DATA / "support_primary_counts.csv", index=False)
positions = np.arange(len(primary_support))
figure, axis = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
width = 0.36
axis.bar(positions - width / 2, primary_support["Source control count"], width, color=PALETTE["cyan"], label="Control")
axis.bar(positions + width / 2, primary_support["Source treated count"], width, color=PALETTE["amber"], label="Treated")
axis.set_xticks(positions, primary_support["Stratum name"])
axis.set_ylabel("Source counties")
axis.set_title("Source Treatment Support in the Three Public Strata")
axis.legend()
finish_axis(axis)
save_figure(figure, "support_primary_counts")

,Requested strata,Realized strata,Target-positive strata,Supported target strata,Unsupported target strata,Smallest source control count,Smallest source treated count,Calculation status
0,2,2,2,2,0,22,11,Pass
1,3,3,3,3,0,5,1,Pass
2,4,4,4,3,1,2,0,Stop
3,5,5,5,4,1,0,0,Stop
4,6,6,6,5,1,0,0,Stop
5,7,7,7,5,2,0,0,Stop
6,8,8,7,5,2,0,0,Stop
7,9,9,8,6,2,0,0,Stop
8,10,10,9,6,3,0,0,Stop


,Requested strata,Supported target strata,Unsupported target strata,Smallest source treated count,Calculation status
0,2,2,0,11,Pass
1,3,3,0,1,Pass
2,4,3,1,0,Stop
3,5,4,1,0,Stop
4,6,5,1,0,Stop
5,7,5,2,0,Stop
6,8,5,2,0,Stop
7,9,6,2,0,Stop
8,10,6,3,0,Stop


,Stratum,Stratum name,Target count,Source control count,Source treated count,Target positive,Supported
0,0,Low,4,116,47,Pass,Pass
1,1,Middle,85,49,32,Pass,Pass
2,2,High,161,5,1,Pass,Pass


,Stratum,Target count,Source control count,Source treated count,Target positive,Supported
0,0,2,90,35,Pass,Pass
1,1,31,58,34,Pass,Pass
2,2,94,20,11,Pass,Pass
3,3,123,2,0,Pass,Fail


,Check,Rejected,Message
0,Four-stratum target standardization,Pass,Target-positive strata lack source treatment s...


<Figure size 880x520 with 1 Axes>

<Figure size 880x520 with 1 Axes>

## Paths Through Target Weights

Let $\pi^S$ and $\pi^T$ be the source and target weight vectors. A path between them is

$$
\pi(\lambda)=(1-\lambda)\pi^S+\lambda\pi^T,\qquad 0\leq\lambda\leq1.
$$

The value $\lambda=0$ gives the source mix, while $\lambda=1$ gives the target mix. Holding the source stratum effects fixed, the standardized effect along the path is

$$
\tau(\lambda)=\sum_{g=1}^{K}\pi_g(\lambda)\tau^S(g).
$$

Thus, $\lambda$ changes population composition, $\pi_g(\lambda)$ is the weight of stratum $g$, and $\tau(\lambda)$ is the resulting weighted contrast. The path does not test whether the source contrasts are transportable.

In [2]:
public_summary = transport_summary(counties, "Stratum")
public_summary["Stratum name"] = public_summary["Stratum"].map({0: "Low", 1: "Middle", 2: "High"})
effects = public_summary["Source contrast"].to_numpy()
source_weights = public_summary["Source probability"].to_numpy()
target_weights = public_summary["Target probability"].to_numpy()

path_rows = []
for mixing_weight in np.linspace(0, 1, 21):
    weights = (1 - mixing_weight) * source_weights + mixing_weight * target_weights
    path_rows.append({
        "Target mixing weight": mixing_weight,
        "Low-stratum probability": weights[0],
        "Middle-stratum probability": weights[1],
        "High-stratum probability": weights[2],
        "Standardized contrast": float(weights @ effects),
    })
weight_path = pd.DataFrame(path_rows)
save_table(weight_path, "support_target_weight_path", "Target-weight path from the public source mix to the public target mix.", "tab:cim6-support-target-weight-path", fit=True)
selected_path = weight_path.iloc[[0, 5, 10, 15, 20]].reset_index(drop=True)
save_table(selected_path, "support_target_weight_path_selected", "Selected points on the public target-weight path.", "tab:cim6-support-target-weight-path-selected", fit=True)
selected_path_book = selected_path.rename(columns={
    "Target mixing weight": "Mix",
    "Low-stratum probability": "Low",
    "Middle-stratum probability": "Middle",
    "High-stratum probability": "High",
    "Standardized contrast": "Contrast",
})
save_table(selected_path_book, "support_target_weight_path_book", "Readable points on the public target-weight path.", "tab:cim6-support-target-weight-path-book")

weight_path.to_csv(FIGURE_DATA / "support_target_weight_path.csv", index=False)
figure, axis = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
axis.plot(weight_path["Target mixing weight"], weight_path["Low-stratum probability"], color=PALETTE["cyan"], linewidth=2.2, label="Low")
axis.plot(weight_path["Target mixing weight"], weight_path["Middle-stratum probability"], color=PALETTE["violet"], linewidth=2.2, label="Middle")
axis.plot(weight_path["Target mixing weight"], weight_path["High-stratum probability"], color=PALETTE["amber"], linewidth=2.2, label="High")
axis.set_xlabel("Target mixing weight")
axis.set_ylabel("Stratum probability")
axis.set_title("A Continuous Path from Source Weights to Target Weights")
axis.legend()
finish_axis(axis)
save_figure(figure, "support_target_weight_path")

effect_path_data = weight_path[["Target mixing weight", "Standardized contrast"]].copy()
effect_path_data.to_csv(FIGURE_DATA / "support_effect_path.csv", index=False)
figure, axis = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
axis.plot(effect_path_data["Target mixing weight"], effect_path_data["Standardized contrast"], color=PALETTE["green"], linewidth=2.5)
axis.axhline(0, color=PALETTE["text"], linewidth=0.9, alpha=0.7)
axis.scatter([0, 1], [effect_path_data.iloc[0]["Standardized contrast"], effect_path_data.iloc[-1]["Standardized contrast"]], color=[PALETTE["cyan"], PALETTE["amber"]], s=70, zorder=3)
axis.set_xlabel("Target mixing weight")
axis.set_ylabel("Standardized contrast")
axis.set_title("The Standardized Contrast Along the Weight Path")
finish_axis(axis)
save_figure(figure, "support_effect_path")

simplex_rows = []
for low_weight in np.linspace(0, 1, 51):
    for middle_weight in np.linspace(0, 1 - low_weight, int(round((1 - low_weight) * 50)) + 1):
        high_weight = 1 - low_weight - middle_weight
        simplex_rows.append({
            "Low-stratum probability": low_weight,
            "Middle-stratum probability": middle_weight,
            "High-stratum probability": high_weight,
            "Standardized contrast": low_weight * effects[0] + middle_weight * effects[1] + high_weight * effects[2],
            "Horizontal coordinate": middle_weight + 0.5 * high_weight,
            "Vertical coordinate": np.sqrt(3) * high_weight / 2,
        })
simplex = pd.DataFrame(simplex_rows)
simplex.to_csv(FIGURE_DATA / "support_weight_simplex.csv", index=False)
simplex_extrema = pd.DataFrame([
    {"Point": "Minimum on grid", **simplex.loc[simplex["Standardized contrast"].idxmin(), ["Low-stratum probability", "Middle-stratum probability", "High-stratum probability", "Standardized contrast"]].to_dict()},
    {"Point": "Maximum on grid", **simplex.loc[simplex["Standardized contrast"].idxmax(), ["Low-stratum probability", "Middle-stratum probability", "High-stratum probability", "Standardized contrast"]].to_dict()},
    {"Point": "Public source mix", "Low-stratum probability": source_weights[0], "Middle-stratum probability": source_weights[1], "High-stratum probability": source_weights[2], "Standardized contrast": float(source_weights @ effects)},
    {"Point": "Public target mix", "Low-stratum probability": target_weights[0], "Middle-stratum probability": target_weights[1], "High-stratum probability": target_weights[2], "Standardized contrast": float(target_weights @ effects)},
])
save_table(simplex_extrema, "support_simplex_extrema", "Selected points on the target-weight simplex.", "tab:cim6-support-simplex-extrema", fit=True)

figure, axis = plt.subplots(figsize=(8.8, 6.2), constrained_layout=True)
scatter = axis.scatter(simplex["Horizontal coordinate"], simplex["Vertical coordinate"], c=simplex["Standardized contrast"], cmap="viridis", s=18)
source_x = source_weights[1] + 0.5 * source_weights[2]
source_y = np.sqrt(3) * source_weights[2] / 2
target_x = target_weights[1] + 0.5 * target_weights[2]
target_y = np.sqrt(3) * target_weights[2] / 2
axis.scatter([source_x], [source_y], color=PALETTE["cyan"], edgecolor=PALETTE["text"], s=95, label="Public source mix")
axis.scatter([target_x], [target_y], color=PALETTE["amber"], edgecolor=PALETTE["text"], s=95, label="Public target mix")
axis.text(-0.02, -0.045, "Low", color=PALETTE["text"])
axis.text(0.98, -0.045, "Middle", color=PALETTE["text"], ha="right")
axis.text(0.5, np.sqrt(3) / 2 + 0.03, "High", color=PALETTE["text"], ha="center")
axis.set_aspect("equal")
axis.set_axis_off()
axis.set_title("Standardized Contrast Across All Three-Stratum Target Mixes")
axis.legend(loc="upper right")
figure.colorbar(scatter, ax=axis, label="Standardized contrast", shrink=0.82)
save_figure(figure, "support_weight_simplex")

,Target mixing weight,Low-stratum probability,Middle-stratum probability,High-stratum probability,Standardized contrast
0,0,0.652,0.324,0.024,-0.035567
1,0.05,0.6202,0.3248,0.055,-0.032658
2,0.1,0.5884,0.3256,0.086,-0.02975
3,0.15,0.5566,0.3264,0.117,-0.026841
4,0.2,0.5248,0.3272,0.148,-0.023933
5,0.25,0.493,0.328,0.179,-0.021024
6,0.3,0.4612,0.3288,0.21,-0.018115
7,0.35,0.4294,0.3296,0.241,-0.015207
8,0.4,0.3976,0.3304,0.272,-0.012298
9,0.45,0.3658,0.3312,0.303,-0.00939


,Target mixing weight,Low-stratum probability,Middle-stratum probability,High-stratum probability,Standardized contrast
0,0,0.652,0.324,0.024,-0.035567
1,0.25,0.493,0.328,0.179,-0.021024
2,0.5,0.334,0.332,0.334,-0.006481
3,0.75,0.175,0.336,0.489,0.008062
4,1,0.016,0.34,0.644,0.022605


,Mix,Low,Middle,High,Contrast
0,0,0.652,0.324,0.024,-0.035567
1,0.25,0.493,0.328,0.179,-0.021024
2,0.5,0.334,0.332,0.334,-0.006481
3,0.75,0.175,0.336,0.489,0.008062
4,1,0.016,0.34,0.644,0.022605


<Figure size 880x520 with 1 Axes>

<Figure size 880x520 with 1 Axes>

,Point,Low-stratum probability,Middle-stratum probability,High-stratum probability,Standardized contrast
0,Minimum on grid,0,1,0,-0.068572
1,Maximum on grid,0,0,1,0.071877
2,Public source mix,0.652,0.324,0.024,-0.035567
3,Public target mix,0.016,0.34,0.644,0.022605


<Figure size 880x620 with 2 Axes>

## Rejecting strata with impossible target mass

If $\pi_g^T=0$, then the stratum contribution is

$$
\pi_g^T\tau^S(g)=0.
$$

The effect $\tau^S(g)$ may be nonzero, but multiplication by the zero target weight removes its contribution. This does not permit nonfinite inputs. If the target weight becomes positive, both source treatment groups are required before the calculation can proceed.

In [3]:
treated_means = np.array([2.0, 4.0, 8.0])
control_means = np.array([1.0, 2.0, 9.0])
effects = treated_means - control_means
zero_weights = np.array([0.25, 0.75, 0.0])
positive_weights = np.array([0.2375, 0.7125, 0.05])
zero_counts_1 = np.array([2.0, 4.0, 0.0])
zero_counts_0 = np.array([3.0, 7.0, 0.0])
supported_counts_1 = np.array([2.0, 4.0, 1.0])
supported_counts_0 = np.array([3.0, 7.0, 1.0])

zero_value = transported_ate(treated_means, control_means, zero_weights, zero_counts_1, zero_counts_0)
positive_value = transported_ate(treated_means, control_means, positive_weights, supported_counts_1, supported_counts_0)
zero_contract = pd.DataFrame([
    {"Case": "Zero target mass and zero source counts", "Target effect": zero_value, "Result": "Pass"},
    {"Case": "Positive target mass with both source groups", "Target effect": positive_value, "Result": "Pass"},
])
save_table(zero_contract, "support_zero_mass_contract", "Zero-target-mass and positive-target-mass calculations.", "tab:cim6-support-zero-mass")

failure_cases = pd.DataFrame([
    capture_failure("Target probabilities do not sum to one", lambda: transported_ate([2, 4], [1, 2], [0.4, 0.5], [2, 2], [2, 2])),
    capture_failure("Negative source count", lambda: transported_ate([2], [1], [1], [-1], [1])),
    capture_failure("Mismatched stratum arrays", lambda: transported_ate([2, 4], [1], [0.5, 0.5], [1, 1], [1, 1])),
    capture_failure("Nonfinite supplied mean", lambda: transported_ate([2, np.nan], [1, 2], [1, 0], [1, 0], [1, 0])),
    capture_failure("Target-positive missing treated support", lambda: transported_ate(treated_means, control_means, positive_weights, zero_counts_1, zero_counts_0)),
])
save_table(failure_cases, "support_invalid_input_checks", "Explicit input and support failures.", "tab:cim6-support-invalid-inputs", fit=True)
failure_cases_book = pd.DataFrame([
    {"Check": "Weights sum to one", "Result": "Stopped as required"},
    {"Check": "Source counts are nonnegative", "Result": "Stopped as required"},
    {"Check": "All stratum lists have equal length", "Result": "Stopped as required"},
    {"Check": "All supplied means are finite", "Result": "Stopped as required"},
    {"Check": "Both source groups appear where the target has mass", "Result": "Stopped as required"},
])
save_table(failure_cases_book, "support_invalid_input_checks_book", "Plain-language input and support checks.", "tab:cim6-support-invalid-inputs-book")
assert failure_cases["Rejected"].all()

zero_contributions = pd.DataFrame({
    "Stratum": [1, 2, 3],
    "Zero-mass contribution": zero_weights * effects,
    "Positive-mass contribution": positive_weights * effects,
})
zero_contributions.to_csv(FIGURE_DATA / "support_zero_mass_contributions.csv", index=False)
positions = np.arange(3)
figure, axis = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
width = 0.36
axis.bar(positions - width / 2, zero_contributions["Zero-mass contribution"], width, color=PALETTE["cyan"], label="Third target weight is zero")
axis.bar(positions + width / 2, zero_contributions["Positive-mass contribution"], width, color=PALETTE["amber"], label="Third target weight is positive")
axis.axhline(0, color=PALETTE["text"], linewidth=0.9, alpha=0.7)
axis.set_xticks(positions, ["Stratum 1", "Stratum 2", "Stratum 3"])
axis.set_ylabel("Contribution to target effect")
axis.set_title("A Zero Target Weight Removes a Stratum Contribution")
axis.legend()
finish_axis(axis)
save_figure(figure, "support_zero_mass_contributions")

support_summary = pd.DataFrame([
    {"Result": "Three public strata", "Value": "All target-positive strata supported"},
    {"Result": "Four public strata", "Value": "One target-positive stratum lacks treated source counties"},
    {"Result": "Zero target mass", "Value": "Zero source counts accepted when supplied means are finite"},
    {"Result": "Positive target mass", "Value": "Both source treatment groups required"},
])
save_table(support_summary, "support_section_summary", "Summary of target weights and support checks.", "tab:cim6-support-summary", fit=True)

,Case,Target effect,Result
0,Zero target mass and zero source counts,1.75,Pass
1,Positive target mass with both source groups,1.6125,Pass


,Check,Rejected,Message
0,Target probabilities do not sum to one,Pass,target_probabilities must have nonnegative row...
1,Negative source count,Pass,Stratum arrays must match with nonnegative sou...
2,Mismatched stratum arrays,Pass,Stratum arrays must match with nonnegative sou...
3,Nonfinite supplied mean,Pass,treated_means must contain finite values and c...
4,Target-positive missing treated support,Pass,Target-positive strata lack source treatment s...


,Check,Result
0,Weights sum to one,Stopped as required
1,Source counts are nonnegative,Stopped as required
2,All stratum lists have equal length,Stopped as required
3,All supplied means are finite,Stopped as required
4,Both source groups appear where the target has...,Stopped as required


<Figure size 880x520 with 1 Axes>

,Result,Value
0,Three public strata,All target-positive strata supported
1,Four public strata,One target-positive stratum lacks treated sour...
2,Zero target mass,Zero source counts accepted when supplied mean...
3,Positive target mass,Both source treatment groups required


,Result,Value
0,Three public strata,All target-positive strata supported
1,Four public strata,One target-positive stratum lacks treated sour...
2,Zero target mass,Zero source counts accepted when supplied mean...
3,Positive target mass,Both source treatment groups required
